In [0]:
%pip install s3fs

In [0]:
dbutils.library.restartPython()

In [0]:
# scripts/generate_raw_data_fixed.py
# Generates messy, realistic raw data across 3 batch days
# Uses Databricks native dbutils for secure S3 writing.

import pandas as pd
import numpy as np
import random
import uuid
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

OUTPUT_DIR = "s3://ecom-landing-zone-2026/landing"

# ─── Databricks Native S3 Uploader (In-Memory) ───────────────────────────────
def save_file(df, folder, filename):
    """Writes DataFrame directly to S3 in-memory using dbutils, bypassing s3fs"""
    s3_path = f"{OUTPUT_DIR}/{folder}/{filename}"
    
    # 1. Convert Pandas DataFrame to a raw string in memory
    if filename.endswith(".json"):
        file_content = df.to_json(orient="records", lines=True)
    else:
        file_content = df.to_csv(index=False)
        
    # 2. Push the string directly to S3 using Databricks native utility
    dbutils.fs.put(s3_path, file_content, True)

# ─── Helper Functions ────────────────────────────────────────────────────────
def random_date(start, end):
    return start + timedelta(days=random.randint(0, (end - start).days))

def corrupt_date(date_str):
    corruption = random.random()
    if corruption < 0.05: return "99/99/9999"  
    elif corruption < 0.10: return date_str.replace("-", "/") 
    elif corruption < 0.13: return None  
    return date_str

def corrupt_amount(amount):
    corruption = random.random()
    if corruption < 0.04: return -abs(amount)  
    elif corruption < 0.07: return None  
    elif corruption < 0.09: return str(amount) + "$"  
    return round(amount, 2)

def add_duplicates(df, pct=0.05):
    n_dupes = int(len(df) * pct)
    dupes = df.sample(n=n_dupes, replace=True)
    return pd.concat([df, dupes], ignore_index=True).sample(frac=1).reset_index(drop=True)

CITIES = ["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai", "Pune", "Kolkata", "Ahmedabad", "Jaipur", "Lucknow"]
STATES = {"Mumbai": "Maharashtra", "Delhi": "Delhi", "Bangalore": "Karnataka", "Hyderabad": "Telangana", "Chennai": "Tamil Nadu", "Pune": "Maharashtra", "Kolkata": "West Bengal", "Ahmedabad": "Gujarat", "Jaipur": "Rajasthan", "Lucknow": "Uttar Pradesh"}
PINCODES = {"Mumbai": [400001, 400050, 400099], "Delhi": [110001, 110020, 110085], "Bangalore": [560001, 560034, 560100], "Hyderabad": [500001, 500032, 500081], "Chennai": [600001, 600028, 600040], "Pune": [411001, 411014, 411038], "Kolkata": [700001, 700029, 700091], "Ahmedabad": [380001, 380015, 380054], "Jaipur": [302001, 302017, 302033], "Lucknow": [226001, 226010, 226022]}
CATEGORIES = ["Electronics", "Clothing", "Books", "Home & Kitchen", "Sports", "Beauty", "Toys", "Automotive", "Groceries"]
ORDER_STATUSES = ["placed", "confirmed", "processing", "shipped", "out_for_delivery", "delivered", "cancelled", "returned"]
PAYMENT_METHODS = ["credit_card", "debit_card", "upi", "net_banking", "cod", "wallet", "CREDIT CARD", "UPI", "COD"]
CUSTOMER_TIERS = ["Bronze", "Silver", "Gold", "Platinum"]
SELLER_TIERS = ["Standard", "Premium", "Elite"]

# ─── Batch Dates ─────────────────────────────────────────────────────────────
BATCH_DATES = [
    datetime(2026, 3, 1),
    datetime(2026, 3, 2),
    datetime(2026, 3, 3),
]

# ─── 1. CUSTOMERS ────────────────────────────────────────────────────────────
def generate_customers(n=200, start_id=1):
    records = []
    for i in range(start_id, start_id + n):
        city = random.choice(CITIES)
        dob = random_date(datetime(1970, 1, 1), datetime(2000, 12, 31))
        tier = random.choice(CUSTOMER_TIERS)
        rec = {
            "customer_id": f"CUST{i:05d}",
            "first_name": random.choice(["Ravi", "Priya", "Arjun", "Sneha", "Kiran", "Ananya", "Rohit", "Meera", "Vikram", "Pooja"]),
            "last_name": random.choice(["Sharma", "Patel", "Kumar", "Singh", "Reddy", "Nair", "Joshi", "Rao", "Gupta", "Mishra"]),
            "email": (None if random.random() < 0.08 else f"user{i}@{'example' if random.random() > 0.05 else 'bad_email_no_at'}.com"),
            "phone": (None if random.random() < 0.06 else f"{'9' if random.random() > 0.05 else '0'}{random.randint(100000000, 999999999)}"),
            "city": city if random.random() > 0.04 else None,
            "state": STATES.get(city, "Unknown") if random.random() > 0.03 else "N/A",
            "pincode": random.choice(PINCODES[city]) if random.random() > 0.05 else None,
            "date_of_birth": corrupt_date(dob.strftime("%Y-%m-%d")),
            "gender": random.choice(["M", "F", "Male", "Female", None, "m", "f"]),
            "customer_tier": tier if random.random() > 0.04 else None,
            "registration_date": random_date(datetime(2020, 1, 1), datetime(2023, 12, 31)).strftime("%Y-%m-%d"),
            "is_active": random.choice([1, 0, "Y", "N", True, False, None]),
            "created_at": datetime(2026, 3, 1).isoformat(),
            "updated_at": datetime(2026, 3, 1).isoformat(),
            "_source_system": "mysql_crm",
            "_batch_id": "batch_2026_03_01"
        }
        records.append(rec)
    df = pd.DataFrame(records)
    return add_duplicates(df, pct=0.06)

# Batch 1: Initial load
customers_b1 = generate_customers(200)
save_file(customers_b1, "customers", "customers_2026_03_01.csv")

# Batch 2: CDC Updates
customers_b2 = generate_customers(200)
update_ids = random.sample([f"CUST{i:05d}" for i in range(1, 201)], 20)
mask = customers_b2["customer_id"].isin(update_ids)
customers_b2.loc[mask, "city"] = random.choice(CITIES)
for idx, row in customers_b2[mask].iterrows():
    customers_b2.at[idx, 'pincode'] = random.choice(PINCODES[row['city']])
customers_b2.loc[mask, "customer_tier"] = random.choice(CUSTOMER_TIERS)
customers_b2.loc[mask, "updated_at"] = datetime(2026, 3, 2).isoformat()
customers_b2.loc[~mask, "_batch_id"] = "batch_2026_03_02"
customers_b2 = customers_b2[mask | (customers_b2["customer_id"].isin([f"CUST{i:05d}" for i in range(201, 231)]))]
new_customers = generate_customers(30, start_id=201)
new_customers["_batch_id"] = "batch_2026_03_02"
customers_b2 = pd.concat([customers_b2[mask], new_customers], ignore_index=True)
customers_b2["_batch_id"] = "batch_2026_03_02"
save_file(customers_b2, "customers", "customers_2026_03_02.csv")

# Batch 3: New customers
customers_b3_new = generate_customers(20, start_id=231)
customers_b3_new["_batch_id"] = "batch_2026_03_03"
save_file(customers_b3_new, "customers", "customers_2026_03_03.csv")
print("✅ Customers generated")


# ─── 2. SELLERS ──────────────────────────────────────────────────────────────
def generate_sellers(n=50):
    records = []
    for i in range(1, n + 1):
        city = random.choice(CITIES)
        rec = {
            "seller_id": f"SELL{i:04d}",
            "seller_name": f"Seller_{random.choice(['Tech', 'Fashion', 'Home', 'Sports', 'Books'])}_{i}",
            "seller_tier": random.choice(SELLER_TIERS),
            "city": city,
            "state": STATES.get(city),
            "gstin": (f"27ABCDE{i:04d}F1Z{random.randint(1, 9)}" if random.random() > 0.07 else None),
            "rating": round(random.uniform(2.5, 5.0), 1) if random.random() > 0.05 else None,
            "total_products": random.randint(10, 500),
            "is_active": random.choice([True, False]),
            "onboarding_date": random_date(datetime(2019, 1, 1), datetime(2023, 6, 1)).strftime("%Y-%m-%d"),
            "commission_pct": round(random.uniform(5, 20), 2),
            "updated_at": datetime(2026, 3, 1).isoformat(),
            "_batch_id": "batch_2026_03_01"
        }
        records.append(rec)
    return pd.DataFrame(records)

sellers = generate_sellers(50)
sellers = add_duplicates(sellers, pct=0.04)
save_file(sellers, "sellers", "sellers_2026_03_01.csv")

sellers_b2 = generate_sellers(50)
update_sellers = random.sample(list(sellers_b2["seller_id"]), 8)
sellers_b2.loc[sellers_b2["seller_id"].isin(update_sellers), "seller_tier"] = "Elite"
sellers_b2.loc[sellers_b2["seller_id"].isin(update_sellers), "updated_at"] = datetime(2026, 3, 2).isoformat()
sellers_b2 = sellers_b2[sellers_b2["seller_id"].isin(update_sellers)]
sellers_b2["_batch_id"] = "batch_2026_03_02"
save_file(sellers_b2, "sellers", "sellers_2026_03_02.csv")
print("✅ Sellers generated")


# ─── 3. PRODUCTS ─────────────────────────────────────────────────────────────
def generate_products(n=500):
    records = []
    for i in range(1, n + 1):
        category = random.choice(CATEGORIES)
        seller_id = f"SELL{random.randint(1, 50):04d}"
        price = round(random.uniform(50, 50000), 2)
        rec = {
            "product_id": f"PROD{i:06d}",
            "product_name": f"{category}_Product_{i}",
            "category": category,
            "sub_category": f"Sub_{random.randint(1, 5)}",
            "brand": random.choice(["BrandA", "BrandB", "BrandC", None, "brand_a"]),
            "seller_id": seller_id,
            "mrp": price,
            "selling_price": corrupt_amount(price * random.uniform(0.6, 1.0)),
            "cost_price": round(price * random.uniform(0.3, 0.7), 2) if random.random() > 0.06 else None,
            "weight_kg": round(random.uniform(0.1, 20), 2) if random.random() > 0.05 else None,
            "is_active": random.choice([True, False, 1, 0]),
            "launch_date": corrupt_date(random_date(datetime(2020, 1, 1), datetime(2023, 12, 31)).strftime("%Y-%m-%d")),
            "updated_at": datetime(2026, 3, 1).isoformat(),
            "_batch_id": "batch_2026_03_01"
        }
        records.append(rec)
    df = pd.DataFrame(records)
    return add_duplicates(df, pct=0.04)

products = generate_products(500)
save_file(products, "products", "products_2026_03_01.csv")

products_b2 = generate_products(500)
price_change_ids = random.sample([f"PROD{i:06d}" for i in range(1, 501)], 50)
mask = products_b2["product_id"].isin(price_change_ids)
products_b2.loc[mask, "selling_price"] = products_b2.loc[mask, "mrp"] * random.uniform(0.7, 0.95)
products_b2.loc[mask, "updated_at"] = datetime(2026, 3, 2).isoformat()
products_b2 = products_b2[mask]
products_b2["_batch_id"] = "batch_2026_03_02"
save_file(products_b2, "products", "products_2026_03_02.csv")
print("✅ Products generated")


# ─── 4. ORDERS ───────────────────────────────────────────────────────────────
def generate_orders(n=2000, batch_date=datetime(2026, 3, 1), batch_id="batch_2026_03_01", existing_order_ids=None):
    records = []
    start_id = int(existing_order_ids[-1].replace("ORD", "")) + 1 if existing_order_ids else 1
    for i in range(start_id, start_id + n):
        customer_id = f"CUST{random.randint(1, 230):05d}"
        order_date = batch_date - timedelta(days=random.randint(0, 2))
        amount = round(random.uniform(100, 50000), 2)
        shipping_city = random.choice(CITIES)
        rec = {
            "order_id": f"ORD{i:08d}",
            "customer_id": customer_id,
            "order_date": corrupt_date(order_date.strftime("%Y-%m-%d")),
            "order_time": order_date.strftime("%H:%M:%S") if random.random() > 0.08 else None,
            "status": random.choice(ORDER_STATUSES) if random.random() > 0.05 else random.choice(ORDER_STATUSES).upper(),
            "total_amount": corrupt_amount(amount),
            "discount_amount": round(amount * random.uniform(0, 0.3), 2) if random.random() > 0.1 else None,
            "final_amount": corrupt_amount(amount * random.uniform(0.7, 1.0)),
            "shipping_address_city": shipping_city if random.random() > 0.07 else None,
            "shipping_address_state": STATES.get(shipping_city, "Unknown") if random.random() > 0.05 else None,
            "shipping_pincode": random.choice(PINCODES[shipping_city]) if random.random() > 0.06 else None,
            "payment_method": random.choice(PAYMENT_METHODS),
            "channel": random.choice(["app", "web", "APP", "WEB", None]),
            "created_at": batch_date.isoformat(),
            "updated_at": batch_date.isoformat(),
            "_source_system": "order_management_system",
            "_batch_id": batch_id
        }
        records.append(rec)
    df = pd.DataFrame(records)
    return add_duplicates(df, pct=0.05)

orders_b1 = generate_orders(2000, BATCH_DATES[0], "batch_2026_03_01")
save_file(orders_b1, "orders", "orders_2026_03_01.csv")

orders_b2_new = generate_orders(1500, BATCH_DATES[1], "batch_2026_03_02", existing_order_ids=[f"ORD{i:08d}" for i in range(1, 2001)])
update_order_ids = random.sample([f"ORD{i:08d}" for i in range(1, 2001)], 300)
update_records = []
for oid in update_order_ids:
    update_shipping_city = random.choice(CITIES)
    update_records.append({
        "order_id": oid,
        "customer_id": f"CUST{random.randint(1, 230):05d}",
        "order_date": BATCH_DATES[0].strftime("%Y-%m-%d"),
        "order_time": BATCH_DATES[0].strftime("%H:%M:%S"),
        "status": random.choice(["confirmed", "processing", "shipped", "cancelled"]),
        "total_amount": round(random.uniform(100, 50000), 2),
        "discount_amount": round(random.uniform(0, 500), 2),
        "final_amount": round(random.uniform(100, 49500), 2),
        "shipping_address_city": update_shipping_city,
        "shipping_address_state": STATES.get(update_shipping_city, "Unknown"),
        "shipping_pincode": random.choice(PINCODES[update_shipping_city]),
        "payment_method": random.choice(["upi", "credit_card", "cod"]),
        "channel": random.choice(["app", "web"]),
        "created_at": BATCH_DATES[0].isoformat(),
        "updated_at": BATCH_DATES[1].isoformat(),
        "_source_system": "order_management_system",
        "_batch_id": "batch_2026_03_02"
    })
orders_b2_updates = pd.DataFrame(update_records)
orders_b2 = pd.concat([orders_b2_new, orders_b2_updates], ignore_index=True)
save_file(orders_b2, "orders", "orders_2026_03_02.csv")

orders_b3 = generate_orders(1000, BATCH_DATES[2], "batch_2026_03_03", existing_order_ids=[f"ORD{i:08d}" for i in range(1, 3501)])
save_file(orders_b3, "orders", "orders_2026_03_03.csv")
print("✅ Orders generated")


# ─── 5. ORDER ITEMS ──────────────────────────────────────────────────────────
def generate_order_items(order_ids, batch_id):
    records = []
    for oid in order_ids:
        for j in range(random.randint(1, 5)):
            price = round(random.uniform(50, 10000), 2)
            qty = random.randint(1, 5)
            rec = {
                "order_item_id": f"{oid}_ITEM{j + 1}",
                "order_id": oid,
                "product_id": f"PROD{random.randint(1, 500):06d}",
                "seller_id": f"SELL{random.randint(1, 50):04d}",
                "quantity": qty if random.random() > 0.04 else None,
                "unit_price": corrupt_amount(price),
                "total_price": corrupt_amount(price * qty),
                "discount_pct": round(random.uniform(0, 40), 2) if random.random() > 0.1 else None,
                "status": random.choice(["active", "cancelled", "returned", None]),
                "created_at": datetime(2026, 3, 1).isoformat(),
                "_batch_id": batch_id
            }
            records.append(rec)
    df = pd.DataFrame(records)
    return add_duplicates(df, pct=0.04)

order_ids_b1 = [f"ORD{i:08d}" for i in range(1, 2001)]
items_b1 = generate_order_items(order_ids_b1, "batch_2026_03_01")
save_file(items_b1, "order_items", "order_items_2026_03_01.csv")

order_ids_b2 = [f"ORD{i:08d}" for i in range(2001, 3501)]
items_b2 = generate_order_items(order_ids_b2, "batch_2026_03_02")
save_file(items_b2, "order_items", "order_items_2026_03_02.csv")
print("✅ Order items generated")


# ─── 6. PAYMENTS ─────────────────────────────────────────────────────────────
def generate_payments(order_ids, batch_id, batch_date):
    records = []
    for oid in order_ids:
        if random.random() > 0.05:
            amount = round(random.uniform(100, 50000), 2)
            rec = {
                "payment_id": f"PAY{oid}",
                "order_id": oid,
                "payment_method": random.choice(PAYMENT_METHODS),
                "payment_status": random.choice(["success", "failed", "pending", "refunded", "SUCCESS", "FAILED"]),
                "amount": corrupt_amount(amount),
                "transaction_id": str(uuid.uuid4()) if random.random() > 0.06 else None,
                "payment_date": corrupt_date(batch_date.strftime("%Y-%m-%d")),
                "gateway": random.choice(["razorpay", "paytm", "stripe", None]),
                "created_at": batch_date.isoformat(),
                "updated_at": batch_date.isoformat(),
                "_batch_id": batch_id
            }
            records.append(rec)
    df = pd.DataFrame(records)
    return add_duplicates(df, pct=0.04)

payments_b1 = generate_payments(order_ids_b1, "batch_2026_03_01", BATCH_DATES[0])
save_file(payments_b1, "payments", "payments_2026_03_01.json")

payments_b2 = generate_payments(order_ids_b2, "batch_2026_03_02", BATCH_DATES[1])
save_file(payments_b2, "payments", "payments_2026_03_02.json")
print("✅ Payments generated")


# ─── 7. SHIPMENTS ────────────────────────────────────────────────────────────
def generate_shipments(order_ids, batch_id, batch_date):
    records = []
    for oid in order_ids:
        if random.random() > 0.08:
            promised_days = random.randint(2, 7)
            actual_days = promised_days + random.randint(-1, 5)
            rec = {
                "shipment_id": f"SHIP{oid}",
                "order_id": oid,
                "carrier": random.choice(["BlueDart", "Delhivery", "DTDC", "FedEx", "Ecom Express", None]),
                "tracking_number": str(uuid.uuid4())[:12].upper() if random.random() > 0.08 else None,
                "shipment_status": random.choice(["dispatched", "in_transit", "out_for_delivery", "delivered", "lost"]),
                "dispatch_date": corrupt_date(batch_date.strftime("%Y-%m-%d")),
                "promised_delivery_date": (batch_date + timedelta(days=promised_days)).strftime("%Y-%m-%d"),
                "actual_delivery_date": ((batch_date + timedelta(days=actual_days)).strftime("%Y-%m-%d") if random.random() > 0.3 else None),
                "delivery_attempts": random.randint(1, 3) if random.random() > 0.05 else None,
                "warehouse_id": f"WH{random.randint(1, 10):02d}",
                "created_at": batch_date.isoformat(),
                "updated_at": batch_date.isoformat(),
                "_batch_id": batch_id
            }
            records.append(rec)
    df = pd.DataFrame(records)
    return add_duplicates(df, pct=0.03)

shipments_b1 = generate_shipments(order_ids_b1, "batch_2026_03_01", BATCH_DATES[0])
save_file(shipments_b1, "shipments", "shipments_2026_03_01.csv")

shipments_b2 = generate_shipments(order_ids_b2, "batch_2026_03_02", BATCH_DATES[1])
save_file(shipments_b2, "shipments", "shipments_2026_03_02.csv")
print("✅ Shipments generated")


# ─── 8. RETURNS ──────────────────────────────────────────────────────────────
def generate_returns(order_ids, batch_id, batch_date):
    records = []
    for oid in random.sample(order_ids, int(len(order_ids) * 0.12)):
        amount = round(random.uniform(100, 10000), 2)
        rec = {
            "return_id": f"RET{oid}",
            "order_id": oid,
            "order_item_id": f"{oid}_ITEM1",
            "return_reason": random.choice(["damaged", "wrong_item", "not_as_described", "changed_mind", None, "DAMAGED"]),
            "return_status": random.choice(["initiated", "picked_up", "received", "refund_processed", "rejected"]),
            "refund_amount": corrupt_amount(amount),
            "return_date": corrupt_date(batch_date.strftime("%Y-%m-%d")),
            "refund_date": ((batch_date + timedelta(days=random.randint(3, 10))).strftime("%Y-%m-%d") if random.random() > 0.4 else None),
            "created_at": batch_date.isoformat(),
            "_batch_id": batch_id
        }
        records.append(rec)
    df = pd.DataFrame(records)
    return add_duplicates(df, pct=0.04)

returns_b1 = generate_returns(order_ids_b1, "batch_2026_03_01", BATCH_DATES[0])
save_file(returns_b1, "returns", "returns_2026_03_01.csv")

returns_b2 = generate_returns(order_ids_b2, "batch_2026_03_02", BATCH_DATES[1])
save_file(returns_b2, "returns", "returns_2026_03_02.csv")
print("✅ Returns generated")


# ─── 9. INVENTORY ────────────────────────────────────────────────────────────
def generate_inventory(batch_date, batch_id):
    records = []
    for i in range(1, 501):
        for wh in range(1, 6):
            stock = random.randint(0, 1000)
            rec = {
                "inventory_id": f"INV_PROD{i:06d}_WH{wh:02d}",
                "product_id": f"PROD{i:06d}",
                "warehouse_id": f"WH{wh:02d}",
                "stock_quantity": stock if random.random() > 0.04 else None,
                "reserved_quantity": random.randint(0, min(stock, 100)) if random.random() > 0.05 else None,
                "reorder_level": random.randint(10, 100),
                "snapshot_date": batch_date.strftime("%Y-%m-%d"),
                "last_updated": batch_date.isoformat(),
                "_batch_id": batch_id
            }
            records.append(rec)
    return pd.DataFrame(records)

inventory_b1 = generate_inventory(BATCH_DATES[0], "batch_2026_03_01")
save_file(inventory_b1, "inventory", "inventory_2026_03_01.csv")

inventory_b2 = generate_inventory(BATCH_DATES[1], "batch_2026_03_02")
save_file(inventory_b2, "inventory", "inventory_2026_03_02.csv")
print("✅ Inventory generated")
print("\n🎉 All raw data files generated successfully using dbutils!")